# 04 — Cross-Sub-Grid Transferability

This notebook evaluates whether the reliability-qualified nighttime lights workflow developed for Samar–Leyte transfers to other Visayas sub-grids. It applies the same GHSL-masked DNB-BRDF preprocessing, spatial-completeness filtering, 30-day smoothing, z-score normalisation, and NTL–NGCP alignment metrics across Samar–Leyte, Bohol, Cebu, Negros, and Panay.

The aim is not to impose one universal threshold. The aim is to test whether the same **selection logic** repeatedly identifies interpretable configurations across different island grids.


## 1. Setup

### 1.1 Import packages

In [ ]:
from pathlib import Path
import glob
import json
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
import sys
import openpyxl
!{sys.executable} -m pip install openpyxl
print(openpyxl.__version__)

### 1.2 Project root and output folders


In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / "datasets"

OUTPUT_MAIN_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "main"
OUTPUT_SUPP_FIG_DIR = PROJECT_ROOT / "outputs" / "figures" / "supplementary"
OUTPUT_MAIN_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "main"
OUTPUT_SUPP_TABLE_DIR = PROJECT_ROOT / "outputs" / "tables" / "supplementary"

for out_dir in [
    OUTPUT_MAIN_FIG_DIR,
    OUTPUT_SUPP_FIG_DIR,
    OUTPUT_MAIN_TABLE_DIR,
    OUTPUT_SUPP_TABLE_DIR,
]:
    out_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Main figures: {OUTPUT_MAIN_FIG_DIR}")
print(f"Supplementary figures: {OUTPUT_SUPP_FIG_DIR}")
print(f"Main tables: {OUTPUT_MAIN_TABLE_DIR}")
print(f"Supplementary tables: {OUTPUT_SUPP_TABLE_DIR}")

### 1.3 Input paths


In [ ]:
NTL_FOLDER = DATASET_DIR / "vnp46a2" / "visayas_ghsl_masked_daily_stats"
NGCP_XLSX_PATH = DATASET_DIR / "ngcp" / "NGCP_hourly_load.xlsx"

print(f"NTL folder: {NTL_FOLDER}")
print(f"NGCP workbook: {NGCP_XLSX_PATH}")


### 1.4 Simple Plotly export helper

Use this same helper across the notebook. It keeps figure display simple and avoids the recurring issue where date axes export as large numeric or exponential values.


In [ ]:
def export_plotly_figure(
    fig,
    output_path,
    width=1200,
    height=600,
    scale=3,
    export_html=True,
    force_date_axis=False,
    tickformat="%Y",
):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig_export = go.Figure(json.loads(pio.to_json(fig, validate=False)))

    if force_date_axis:
        fig_export.update_xaxes(type="date", tickformat=tickformat)

    fig_export.write_image(
        str(output_path.with_suffix(".png")),
        width=width,
        height=height,
        scale=scale,
    )

    fig_export.write_image(
        str(output_path.with_suffix(".svg")),
        width=width,
        height=height,
        scale=scale,
    )

    if export_html:
        fig_export.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs="cdn",
            full_html=True,
        )

    print(f"Exported: {output_path.with_suffix('.png')}")
    print(f"Exported: {output_path.with_suffix('.svg')}")
    if export_html:
        print(f"Exported: {output_path.with_suffix('.html')}")

### 1.5 Shared constants and display metadata


In [ ]:
DATE_START = pd.Timestamp("2012-01-19")
DATE_END = pd.Timestamp("2024-12-31")
ANALYSIS_DAYS = (DATE_END - DATE_START).days + 1

VALUE_COL = "Mean_by_sum"
ROLLING_WINDOW = 30
THRESHOLD_LIST = list(range(10, 100, 10))

STORM_MARKERS = [
    ("Haiyan", "2013-11-08"),
    ("Rai", "2021-12-16"),
]

REGION_RENAME = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}

REGION_ORDER = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]

REGION_COLORS = {
    "Bohol": "#4E79A7",
    "Cebu": "#D65A4A",
    "Negros": "#5AB87A",
    "Panay": "#8E66C6",
    "Samar-Leyte": "#F1A340",
}

GROUP_RENAME = {
    "30-only": "30",
}

GHSL_META = {
    "30": {"code": "G1", "label": "Urban Core", "color": "#ff0000"},
    "23-30": {"code": "G2", "label": "Dense Urban", "color": "#732600"},
    "22-30": {"code": "G3", "label": "Core & Inner", "color": "#a87000"},
    "21-30": {"code": "G4", "label": "Urban Fringe", "color": "#ffff00"},
    "13-30": {"code": "G5", "label": "Built-Up / Rural Mix", "color": "#375623"},
    "12-30": {"code": "G6", "label": "Low–Mid Rural", "color": "#abcd66"},
    "11-30": {"code": "G7", "label": "Rural Settlements", "color": "#cdf57a"},
    "10-30": {"code": "G8", "label": "All Land", "color": "#7ab6f5"},
}

GHSL_ORDER = ["30", "23-30", "22-30", "21-30", "13-30", "12-30", "11-30", "10-30"]

CLASS_ORDER = {
    "30": 1,
    "23-30": 2,
    "22-30": 3,
    "21-30": 4,
    "13-30": 5,
    "12-30": 6,
    "11-30": 7,
    "10-30": 8,
}

### 1.6 Shared helper functions


In [ ]:
# ==============================
# Shared Helpers
# ==============================

def date_str_series(x):
    return pd.to_datetime(x).dt.strftime("%Y-%m-%d")


def add_event_markers(fig, y=1.04):
    for name, event_date in STORM_MARKERS:
        fig.add_vline(
            x=event_date,
            line_dash="dash",
            line_color="black",
            line_width=2,
        )

        fig.add_annotation(
            x=event_date,
            y=y,
            yref="paper",
            text=name,
            showarrow=False,
            font=dict(size=14, color="black"),
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            xanchor="left",
        )

    return fig


def zscore(series):
    s = pd.Series(series).astype(float)
    sd = s.std(ddof=0)

    if sd == 0 or np.isnan(sd):
        return pd.Series(np.nan, index=s.index)

    return (s - s.mean()) / sd


def hex_to_rgba(hex_color, alpha=0.7):
    hex_color = hex_color.lstrip("#")
    r, g, b = tuple(int(hex_color[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def ghsl_label(group):
    meta = GHSL_META.get(group, {})
    code = meta.get("code", group)
    label = meta.get("label", "")
    return f"{code}: {label}" if label else code

## 2. Load and Assemble Multi-Region GHSL-Masked NTL

### 2.1 Parse filenames

Each CSV represents one region and one GHSL cumulative mask. The filename contains both identifiers.

In [ ]:
def parse_filename(filepath):
    name = Path(filepath).name.replace("_stats.csv", "")
    parts = name.split("_DNBBRDF_class")

    if len(parts) != 2:
        return "Unknown", "Unknown"

    region, group = parts
    group = GROUP_RENAME.get(group, group)
    region = REGION_RENAME.get(region, region)

    return region, group

### 2.2 Load regional GHSL-masked NTL files

This section assembles all region × GHSL-mask daily NTL exports into one long table.

Expected output columns include:

| Column | Description |
|---|---|
| `date` | Daily observation date. |
| `Region` | NGCP sub-grid / island region. |
| `Group` | GHSL cumulative thematic mask. |
| `ClassID` | Ordered mask identifier from G1 to G8. |
| `Mean_by_sum` | Mean DNB-BRDF radiance metric used for alignment. |
| `Valid_px` | Number of valid observed pixels in the mask. |
| `NTL_mean`, `NTL_median`, `NTL_min`, `NTL_max` | Supporting DNB-BRDF summary statistics. |


In [ ]:
# ==============================
# Load Multi-Region NTL Panel
# ==============================

csv_files = sorted(glob.glob(str(NTL_FOLDER / "*.csv")))
print(f"Found {len(csv_files)} CSV files")

df_list = []

for file in tqdm(csv_files):
    region, group = parse_filename(file)

    try:
        temp_df = pd.read_csv(file)
    except Exception:
        try:
            temp_df = pd.read_csv(file, dtype=str, engine="python", on_bad_lines="skip")
        except Exception:
            print(f"Skipping unreadable file: {file}")
            continue

    temp_df["Region"] = region
    temp_df["Group"] = group
    df_list.append(temp_df)

if not df_list:
    raise RuntimeError("No NTL CSV files were loaded. Check NTL_FOLDER.")

df_all = pd.concat(df_list, ignore_index=True)

df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
df_all = df_all[(df_all["date"] >= DATE_START) & (df_all["date"] <= DATE_END)].copy()

df_all["Group"] = df_all["Group"].replace(GROUP_RENAME)
df_all["Region"] = df_all["Region"].replace(REGION_RENAME)
df_all["ClassID"] = df_all["Group"].map(CLASS_ORDER)

numeric_cols = ["Mean_by_sum", "NTL_max", "NTL_mean", "NTL_median", "NTL_min", "NTL_std", "Valid_px"]

for col in numeric_cols:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors="coerce")

df_all = df_all.sort_values(["Region", "ClassID", "date"]).reset_index(drop=True)

df_all.head()

In [ ]:
# ==============================
# Export Supplementary Table: Input Inventory
# ==============================

input_inventory = (
    df_all.groupby(["Region", "Group", "ClassID"], as_index=False)
    .agg(
        start_date=("date", "min"),
        end_date=("date", "max"),
        n_rows=("date", "size"),
        valid_rows=("Valid_px", lambda s: int((s > 0).sum())),
    )
)

input_inventory.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_cross_subgrid_ntl_input_inventory.csv",
    index=False,
)

input_inventory

### 2.3 Assembly check

The assembled table should contain five regions and eight GHSL masks per region.

In [ ]:
# ==============================
# Compact Assembly Check
# ==============================

print(f"Rows: {len(df_all):,}")
print(f"Regions: {sorted(df_all['Region'].dropna().unique())}")
print(f"Groups: {sorted(df_all['Group'].dropna().unique(), key=lambda g: CLASS_ORDER.get(g, 99))}")

df_all.groupby(["Region", "Group"]).size().unstack("Group")

## 3. Regional NTL Time Series Diagnostics

### 3.1 Region-specific GHSL time-series viewer

This plot is an exploratory diagnostic. It allows inspection of spatial completeness and 30-day NTL dynamics for one region at a time.

The top panel displays daily spatial completeness. The lower panel displays threshold-qualified DNB-BRDF radiance and the 30-day moving average for the selected GHSL mask.


In [ ]:
# ==============================
# Region-Specific NTL Time Series
# ==============================

def plot_region_ntl_timeseries(df_all, region, tau=40, w=30, value_col=VALUE_COL):
    df_region = df_all[df_all["Region"] == region].copy()

    if df_region.empty:
        raise ValueError(f"No NTL data found for region: {region}")

    groups = [g for g in GHSL_ORDER if g in df_region["Group"].unique()]

    fig = make_subplots(
        rows=2,
        cols=1,
        row_heights=[0.12, 0.78],
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=("Daily spatial completeness, SC (%)", "Mean DNB-BRDF radiance"),
    )

    buttons = []
    n_traces_per_group = 3

    for i, group in enumerate(groups):
        gdf = df_region[df_region["Group"] == group].copy()
        gdf = gdf[gdf["Valid_px"] > 0].copy().sort_values("date")

        if gdf.empty:
            continue

        hi = np.nanpercentile(gdf[value_col], 95)
        gdf = gdf[gdf[value_col] <= hi].copy()

        total_px = gdf["Valid_px"].max()
        gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
        gdf["date_plot"] = date_str_series(gdf["date"])

        fig.add_trace(
            go.Heatmap(
                z=np.expand_dims(gdf["SC_pct"].values, axis=0),
                x=gdf["date_plot"],
                y=[0],
                colorscale="Greens",
                zmin=0,
                zmax=100,
                showscale=True,
                colorbar=dict(title="SC (%)", len=0.25, thickness=18, x=0.95, y=0.95),
                visible=(i == 0),
            ),
            row=1,
            col=1,
        )

        gdf = gdf[gdf["SC_pct"] > tau].copy()

        if not gdf.empty:
            gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=1).mean()
            gdf["date_plot"] = date_str_series(gdf["date"])

        group_color = GHSL_META.get(group, {}).get("color", "#999999")
        group_code = GHSL_META.get(group, {}).get("code", group)

        fig.add_trace(
            go.Scatter(
                x=gdf["date_plot"],
                y=gdf[value_col],
                mode="markers",
                marker=dict(color=group_color, size=4, opacity=0.18),
                name=f"{group_code} daily",
                visible=(i == 0),
                showlegend=False,
            ),
            row=2,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=gdf["date_plot"],
                y=gdf[f"MA_{w}d"],
                mode="lines",
                line=dict(color=group_color, width=2.2),
                name=f"{group_code} {w}d MA",
                visible=(i == 0),
            ),
            row=2,
            col=1,
        )

        visibility = [False] * (n_traces_per_group * len(groups))
        for j in range(n_traces_per_group):
            visibility[i * n_traces_per_group + j] = True

        buttons.append(
            dict(
                label=group_code,
                method="update",
                args=[
                    {"visible": visibility},
                    {"title": f"{region} DNB-BRDF NTL — {group_code} (SC > {tau}%)"},
                ],
            )
        )

    add_event_markers(fig, y=0.86)

    fig.update_layout(
        updatemenus=[dict(type="buttons", buttons=buttons, direction="right", showactive=True, x=0.5, xanchor="center", y=-0.35, yanchor="top")],
        xaxis2=dict(title="", rangeslider=dict(visible=True), type="date"),
        yaxis=dict(showticklabels=False),
        yaxis2=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
        template="plotly_white",
        paper_bgcolor="rgba(0,0,0,0)",
        legend=dict(orientation="h", y=0.88, x=0.8, font=dict(size=16)),
        height=650,
        width=1200,
        margin=dict(t=70, l=70, b=110, r=40),
    )

    fig.update_xaxes(tickfont=dict(size=14))
    fig.update_yaxes(tickfont=dict(size=14))

    return fig

In [ ]:
fig_sx_region_ntl = plot_region_ntl_timeseries(df_all, region="Samar-Leyte", tau=40, w=ROLLING_WINDOW)
fig_sx_region_ntl.show()

fig_sx_region_ntl.write_html(
    OUTPUT_SUPP_FIG_DIR / "fig_sx_region_ntl_timeseries_samar_leyte.html",
    include_plotlyjs="cdn",
    full_html=True,
)

### 3.2 Interpretation

The regional NTL viewer is useful for checking whether the same GHSL mask behaves similarly across islands. It shows that spatial completeness varies strongly through time, and that thresholding directly changes which portions of the radiance trajectory remain available.

This plot is retained as a supplementary exploratory output rather than a main manuscript figure.


## 4. Regional Comparison Within Each GHSL Mask

### 4.1 Compare regions by selected GHSL group

This figure compares 30-day DNB-BRDF trajectories across regions for a selected GHSL group. The buttons toggle the GHSL mask.


In [ ]:
# ==============================
# Regional Comparison by GHSL Group
# ==============================

def plot_ma_by_class_group(df_all, tau=30, w=30, value_col=VALUE_COL):
    groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]
    regions = [r for r in REGION_ORDER if r in df_all["Region"].unique()]

    fig = go.Figure()
    buttons = []
    n_traces_per_group = len(regions)

    for i, group in enumerate(groups):
        gcode = GHSL_META.get(group, {}).get("code", group)
        gdesc = GHSL_META.get(group, {}).get("label", "")

        for region in regions:
            gdf = df_all[(df_all["Group"] == group) & (df_all["Region"] == region)].copy()
            gdf = gdf[gdf["Valid_px"] > 0].sort_values("date")

            if gdf.empty:
                continue

            hi = np.nanpercentile(gdf[value_col], 95)
            gdf = gdf[gdf[value_col] <= hi].copy()

            total_px = gdf["Valid_px"].max()
            gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
            gdf = gdf[gdf["SC_pct"] > tau].copy()

            if gdf.empty:
                continue

            gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=w).mean()
            gdf["date_plot"] = date_str_series(gdf["date"])

            fig.add_trace(
                go.Scatter(
                    x=gdf["date_plot"],
                    y=gdf[f"MA_{w}d"],
                    mode="lines",
                    name=region,
                    line=dict(color=REGION_COLORS.get(region), width=2),
                    visible=(i == 0),
                )
            )

        visibility = [False] * (n_traces_per_group * len(groups))
        for j in range(n_traces_per_group):
            visibility[i * n_traces_per_group + j] = True

        buttons.append(
            dict(
                label=gcode,
                method="update",
                args=[{"visible": visibility}, {"title": f"{gcode}: {gdesc} — SC > {tau}%"}],
            )
        )

    add_event_markers(fig, y=0.88)

    fig.update_layout(
        xaxis=dict(title="", rangeslider=dict(visible=True), type="date"),
        yaxis=dict(title="Radiance (nW·cm⁻²·sr⁻¹)"),
        updatemenus=[dict(type="buttons", buttons=buttons, direction="right", showactive=True, x=0.5, xanchor="center", y=-0.35, yanchor="top")],
        legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center", font=dict(size=14)),
        template="plotly_white",
        height=600,
        width=1200,
        margin=dict(t=60, l=70, b=100, r=40),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig

In [ ]:
fig_sx_region_by_group = plot_ma_by_class_group(df_all, tau=30, w=ROLLING_WINDOW)
fig_sx_region_by_group.show()

fig_sx_region_by_group.write_html(
    OUTPUT_SUPP_FIG_DIR / "fig_sx_region_comparison_by_ghsl_group.html",
    include_plotlyjs="cdn",
    full_html=True,
)

### 4.2 Interpretation

The regional comparison helps separate two sources of transferability: settlement structure and regional context. If urban masks behave similarly across regions, this supports transferability of the settlement-stratified logic. If the same threshold produces different temporal coverage or signal shape, this indicates that the admissibility threshold remains region-specific.

## 5. Load NGCP Sub-Grid Electricity Load

### 5.1 Load 1 AM NGCP load from workbook

This section loads 1 AM electricity load for all five Visayas sub-grids. The 1 AM value is used to approximate the daily load closest to the VIIRS nighttime overpass.


In [ ]:
# ==============================
# Load NGCP 1 AM Load
# ==============================

def load_ngcp_grid_1am_flat(
    xlsx_path,
    sheet_map,
    date_start=DATE_START,
    date_end=DATE_END,
):
    data = {}

    for region, sheet_name in sheet_map.items():
        df = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=1)

        # Standardize column names
        df.columns = [str(c).strip() for c in df.columns]

        # Find date column
        date_col = df.columns[0]
        df = df.rename(columns={date_col: "DATE"})

        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")
        df = df.dropna(subset=["DATE"]).copy()
        df = df.set_index("DATE")

        # Clean column labels again after DATE indexing
        df.columns = [str(c).strip() for c in df.columns]

        # Try common 1 AM column names first
        possible_hour_cols = ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]

        hour_col = next(
            (c for c in possible_hour_cols if c in df.columns),
            None,
        )

        # Fallback: use the first numeric-looking hourly column
        if hour_col is None:
            numeric_like_cols = [
                c for c in df.columns
                if str(c).replace(".0", "").isdigit()
            ]

            if numeric_like_cols:
                hour_col = numeric_like_cols[0]

        # Final fallback: use first column after DATE
        if hour_col is None and len(df.columns) > 0:
            hour_col = df.columns[0]
            print(f"Using fallback column for {region}: {hour_col}")

        if hour_col is None:
            raise KeyError(f"No usable load column found in sheet: {sheet_name}")

        s = pd.to_numeric(df[hour_col], errors="coerce")
        s = s[(s.index >= date_start) & (s.index <= date_end)]

        data[region] = s

        print(f"{region}: using column '{hour_col}' from sheet '{sheet_name}'")

    return pd.DataFrame(data).sort_index()


SHEET_MAP = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

ngcp_df = load_ngcp_grid_1am_flat(NGCP_XLSX_PATH, SHEET_MAP)
ngcp_df.head()

In [ ]:
ngcp_df.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_ngcp_1am_load_visayas_2012_2024.csv",
    index=True,
)


## 6. NGCP Load Dynamics

### 6.1 Combined sub-grid load time series

This figure summarises the 30-day moving average of 1 AM NGCP load for all five sub-grids.


In [ ]:
# ==============================
# NGCP Combined 30-Day Moving Average
# ==============================

def plot_ngcp_combined_ma(df_grid, w=30):
    df = df_grid.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df[(df.index >= DATE_START) & (df.index <= DATE_END)]

    fig = go.Figure()

    for region in [r for r in REGION_ORDER if r in df.columns]:
        ma = df[region].rolling(window=w, min_periods=w).mean()

        fig.add_trace(
            go.Scatter(
                x=[d.strftime("%Y-%m-%d") for d in ma.index],
                y=ma,
                mode="lines",
                name=region,
                line=dict(width=2.5, color=REGION_COLORS.get(region)),
            )
        )

    add_event_markers(fig, y=0.88)

    fig.update_layout(
        template="plotly_white",
        height=600,
        width=1200,
        xaxis=dict(title="Date", type="date", rangeslider=dict(visible=False)),
        yaxis=dict(title="Grid load (MW)"),
        legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center", font=dict(size=14)),
        margin=dict(t=80, l=70, b=80, r=40),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig

In [ ]:
fig_sx_ngcp_load = plot_ngcp_combined_ma(ngcp_df, w=ROLLING_WINDOW)
fig_sx_ngcp_load.show()

export_plotly_figure(
    fig_sx_ngcp_load,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_ngcp_subgrid_load_ma30_2012_2024",
    width=1200,
    height=600,
    scale=3,
    force_date_axis=True,
    tickformat="%Y",
)

### 6.2 Interpretation

The NGCP time series provide the independent ground reference for the transferability assessment. Differences in load magnitude and trend among sub-grids are expected because each grid serves a different settlement and demand structure. The later z-score alignment removes these scale differences so that comparison focuses on timing and shape rather than absolute units.


## 7. Region-Specific NTL–NGCP Z-Score Alignment

### 7.1 Plot GHSL classes against regional NGCP load

This function compares settlement-stratified DNB-BRDF radiance with NGCP load for a selected sub-grid. It uses the same processing logic as the Samar–Leyte alignment.

In [ ]:
# ==============================
# Region-Specific NTL–NGCP Alignment Plot
# ==============================

def plot_ghsl_classes_vs_ngcp_zscore(
    df_all,
    ngcp_df,
    region="Samar-Leyte",
    ghsl_groups=None,
    tau=40,
    w=30,
    value_col=VALUE_COL,
    total_possible_days=ANALYSIS_DAYS,
    legend_position="right",
):
    if ghsl_groups is None:
        ghsl_groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]

    ghsl_groups_sorted = sorted(
        ghsl_groups,
        key=lambda g: GHSL_META.get(g, {}).get("code", "G9"),
        reverse=True,
    )

    if region not in ngcp_df.columns:
        raise ValueError(f"Region '{region}' not found in NGCP columns.")

    ng = ngcp_df[[region]].copy().rename(columns={region: "Grid_1AM_MW"})
    ng.index = pd.to_datetime(ng.index)
    ng = ng.sort_index()
    ng["Load_MA"] = ng["Grid_1AM_MW"].rolling(w, min_periods=1).mean()

    fig = go.Figure()

    for group in ghsl_groups_sorted:
        gdf = df_all[(df_all["Group"] == group) & (df_all["Region"] == region)].copy()
        gdf = gdf[gdf["Valid_px"] > 0].sort_values("date")

        if gdf.empty:
            continue

        hi = np.nanpercentile(gdf[value_col], 95)
        gdf = gdf[gdf[value_col] <= hi].copy()

        total_px = gdf["Valid_px"].max()
        gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
        gdf = gdf[gdf["SC_pct"] > tau].copy()

        if gdf.empty:
            continue

        gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=1).mean()
        ng_merge = ng.reset_index().rename(columns={ng.index.name or "index": "Date"})

        merged = pd.merge(
            gdf.rename(columns={"date": "Date"})[["Date", f"MA_{w}d"]],
            ng_merge[["Date", "Load_MA"]],
            on="Date",
            how="inner",
        ).dropna()

        if merged.empty:
            continue

        ntl_z = zscore(merged[f"MA_{w}d"])
        load_z = zscore(merged["Load_MA"])

        r = np.corrcoef(ntl_z, load_z)[0, 1]
        rmse = np.sqrt(np.mean((load_z - ntl_z) ** 2))
        n = len(merged)
        coverage_pct = n / total_possible_days * 100

        gcode = GHSL_META.get(group, {}).get("code", group)
        glabel = GHSL_META.get(group, {}).get("label", "")
        legend_label = f"{gcode}: {glabel}<br>r={r:.2f}, RMSE={rmse:.2f}, τ={tau}%, N={coverage_pct:.0f}%"

        line_width = 3.5 if gcode in ("G1", "G2", "G3") else 2
        alpha = 0.9 if gcode in ("G1", "G2", "G3") else 0.5
        group_color = GHSL_META.get(group, {}).get("color", "#999999")

        fig.add_trace(
            go.Scatter(
                x=date_str_series(merged["Date"]),
                y=ntl_z,
                mode="lines",
                line=dict(color=hex_to_rgba(group_color, alpha), width=line_width),
                showlegend=False,
            )
        )

        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="lines",
                name=legend_label,
                line=dict(color=group_color, width=6),
                showlegend=True,
            )
        )

    ngcp_z = zscore(ng["Load_MA"])

    fig.add_trace(
        go.Scatter(
            x=[d.strftime("%Y-%m-%d") for d in ng.index],
            y=ngcp_z,
            name="NGCP Electric Load",
            mode="lines",
            line=dict(color="black", width=5),
        )
    )

    add_event_markers(fig, y=1.05)

    if legend_position == "top":
        legend_cfg = dict(orientation="h", x=0.02, y=1.55, xanchor="left", font=dict(size=14))
        width = 850
        height = 700
        margin = dict(t=80, b=70, l=80, r=40)
    else:
        legend_cfg = dict(orientation="v", x=1.02, y=1, font=dict(size=13))
        width = 1200
        height = 520
        margin = dict(t=60, b=70, l=80, r=360)

    fig.update_layout(
        # title=f"{region}: reliability-qualified NTL vs NGCP load",
        xaxis=dict(title="Date", type="date"),
        yaxis=dict(title="Normalised units (z-score)"),
        legend=legend_cfg,
        template="plotly_white",
        height=height,
        width=width,
        margin=margin,
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig

In [ ]:
# ==============================
# Export Region-Specific Supplementary Alignment Figures
# ==============================

example_region_settings = {
    "Samar-Leyte": 10,
    "Bohol": 20,
    "Cebu": 60,
    "Negros": 10,
    "Panay": 10,
}

for region, tau in example_region_settings.items():
    fig_region = plot_ghsl_classes_vs_ngcp_zscore(
        df_all,
        ngcp_df,
        region=region,
        tau=tau,
        w=ROLLING_WINDOW,
        legend_position="top",
    )

    fig_region.show()

    safe_region = region.lower().replace("-", "_").replace(" ", "_")

    export_plotly_figure(
        fig_region,
        OUTPUT_SUPP_FIG_DIR / f"fig_sx_{safe_region}_zscore_alignment_tau{tau}",
        width=850,
        height=700,
        scale=3,
        force_date_axis=True,
        tickformat="%Y",
    )


### 7.2 Interpretation

The z-score plots show whether the timing and shape of settlement-stratified NTL follow the corresponding NGCP load series. Across regions, urban-focused masks generally provide the most interpretable load-aligned signal, while broader masks remain weaker or more diffuse.

The region-specific plots are not used as the primary transferability result. They provide visual support for the composite ranking that follows.


## 8. Composite Performance Score

### 8.1 Rationale

This section ranks every `Region × GHSL Group × τ` configuration. The score balances:

| Component | Direction | Role |
|---|---|---|
| Pearson `r` | Higher is better | Measures shape agreement between NTL and NGCP load. |
| RMSE | Lower is better | Penalises mismatch in normalised units. |
| Temporal coverage | Higher is better | Rewards configurations that retain more dates. |

The score is normalised within each sub-grid. Therefore, composite scores rank configurations **within** a region and should not be interpreted as direct performance values across regions.


In [ ]:
# ==============================
# Alignment Metrics Collection
# ==============================

def collect_ghsl_ngcp_metrics(
    df_all,
    ngcp_df,
    regions=None,
    threshold_list=THRESHOLD_LIST,
    w=30,
    value_col=VALUE_COL,
    total_possible_days=ANALYSIS_DAYS,
):
    ghsl_groups = [g for g in GHSL_ORDER if g in df_all["Group"].unique()]

    if regions is None:
        regions = [r for r in REGION_ORDER if r in df_all["Region"].unique()]

    results = []

    for region in regions:
        if region not in ngcp_df.columns:
            continue

        ng = ngcp_df[[region]].copy().rename(columns={region: "Grid_1AM_MW"})
        ng.index = pd.to_datetime(ng.index)
        ng = ng.sort_index()
        ng["Load_MA"] = ng["Grid_1AM_MW"].rolling(w, min_periods=1).mean()
        ng_merge = ng.reset_index().rename(columns={ng.index.name or "index": "Date"})

        for group in tqdm(ghsl_groups, desc=f"Processing {region}", unit="group"):
            for tau in threshold_list:
                gdf = df_all[(df_all["Group"] == group) & (df_all["Region"] == region)].copy()
                gdf = gdf[gdf["Valid_px"] > 0].sort_values("date")

                if gdf.empty:
                    continue

                hi = np.nanpercentile(gdf[value_col], 95)
                gdf = gdf[gdf[value_col] <= hi].copy()

                total_px = gdf["Valid_px"].max()
                gdf["SC_pct"] = gdf["Valid_px"] / total_px * 100
                gdf = gdf[gdf["SC_pct"] > tau].copy()

                if gdf.empty:
                    continue

                gdf[f"MA_{w}d"] = gdf[value_col].rolling(w, min_periods=1).mean()

                merged = pd.merge(
                    gdf.rename(columns={"date": "Date"})[["Date", f"MA_{w}d"]],
                    ng_merge[["Date", "Load_MA"]],
                    on="Date",
                    how="inner",
                ).dropna()

                if len(merged) < 5:
                    continue

                ntl_z = zscore(merged[f"MA_{w}d"])
                load_z = zscore(merged["Load_MA"])

                r = np.corrcoef(ntl_z, load_z)[0, 1]
                rmse = np.sqrt(np.mean((load_z - ntl_z) ** 2))
                n_days = len(merged)
                coverage_pct = n_days / total_possible_days * 100

                meta = GHSL_META.get(group, {})
                gcode = meta.get("code", group)
                glabel = meta.get("label", "")
                thematic_mask = f"{gcode}: {glabel}" if glabel else gcode

                results.append(
                    {
                        "Region": region,
                        "Group": group,
                        "Thematic_Mask": thematic_mask,
                        "Class_Order": int(gcode.replace("G", "")) if gcode.startswith("G") else np.nan,
                        "tau": tau,
                        "r": r,
                        "rmse": rmse,
                        "n_days": n_days,
                        "coverage_pct": coverage_pct,
                    }
                )

    return pd.DataFrame(results)


def compute_composite_score(df, groupby_cols=["Region"], w_r=0.6, w_rmse=0.1, w_coverage=0.3):
    if not np.isclose(w_r + w_rmse + w_coverage, 1.0):
        raise ValueError("Composite-score weights must sum to 1.")

    df_scored = df.copy()

    df_scored["r_score"] = df_scored.groupby(groupby_cols)["r"].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
    )

    df_scored["rmse_score"] = df_scored.groupby(groupby_cols)["rmse"].transform(
        lambda x: 1 - (x - x.min()) / (x.max() - x.min() + 1e-9)
    )

    df_scored["coverage_score"] = df_scored.groupby(groupby_cols)["coverage_pct"].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
    )

    df_scored["composite_score"] = (
        w_r * df_scored["r_score"]
        + w_rmse * df_scored["rmse_score"]
        + w_coverage * df_scored["coverage_score"]
    )

    return df_scored

### 8.2 Compute and rank configurations


In [ ]:
# ==============================
# Compute Cross-Sub-Grid Metrics
# ==============================

df_metrics = collect_ghsl_ngcp_metrics(
    df_all,
    ngcp_df,
    regions=REGION_ORDER,
    threshold_list=THRESHOLD_LIST,
    w=ROLLING_WINDOW,
    value_col=VALUE_COL,
)

df_ranked = compute_composite_score(
    df_metrics,
    groupby_cols=["Region"],
    w_r=0.6,
    w_rmse=0.1,
    w_coverage=0.3,
)

df_ranked = (
    df_ranked
    .sort_values(
        ["Region", "composite_score", "coverage_pct", "r", "rmse"],
        ascending=[True, False, False, False, True],
    )
    .reset_index(drop=True)
)

df_ranked.head()

In [ ]:
# ==============================
# Export Supplementary Full Metrics
# ==============================

df_metrics.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_cross_subgrid_alignment_metrics_all.csv",
    index=False,
)

df_ranked.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_sx_cross_subgrid_alignment_metrics_ranked.csv",
    index=False,
)

## 9. Main Table 4 and Supplementary Table S10

### 9.1 Top five configurations per sub-grid

This table is the expanded supplementary ranking. It shows the five highest-scoring configurations within each sub-grid.


In [ ]:
# ==============================
# Top Five Configurations per Sub-Grid
# ==============================

top5_by_region = (
    df_ranked
    .groupby("Region", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top5_by_region["Rank"] = top5_by_region.groupby("Region").cumcount() + 1

table_s10 = top5_by_region[
    [
        "Region",
        "Rank",
        "Thematic_Mask",
        "Group",
        "tau",
        "composite_score",
        "r",
        "rmse",
        "n_days",
        "coverage_pct",
    ]
].copy()

table_s10 = table_s10.rename(
    columns={
        "Region": "Sub-grid",
        "Thematic_Mask": "Thematic Mask",
        "Group": "GHSL Range",
        "tau": "τ (%)",
        "composite_score": "Composite Score",
        "rmse": "RMSE",
        "n_days": "N days",
        "coverage_pct": "Temporal Coverage (%)",
    }
)

In [ ]:
table_s10_display = table_s10.copy()
table_s10_display["Composite Score"] = table_s10_display["Composite Score"].round(3)
table_s10_display["r"] = table_s10_display["r"].round(2)
table_s10_display["RMSE"] = table_s10_display["RMSE"].round(2)
table_s10_display["Temporal Coverage (%)"] = table_s10_display["Temporal Coverage (%)"].round(2)

table_s10_display

In [ ]:
# ==============================
# Export Supplementary Table S10
# ==============================

table_s10_display.to_csv(
    OUTPUT_SUPP_TABLE_DIR / "table_s10_top5_cross_subgrid_alignment_configurations.csv",
    index=False,
)

### 9.2 Highest-scoring configuration per sub-grid

This is the main manuscript output for Notebook 4.

In [ ]:
# ==============================
# Main Table 4: Highest-Scoring Configuration per Sub-Grid
# ==============================

best_by_region = (
    df_ranked
    .groupby("Region", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

table_04 = best_by_region[
    [
        "Region",
        "Thematic_Mask",
        "Group",
        "tau",
        "composite_score",
        "r",
        "rmse",
        "n_days",
        "coverage_pct",
    ]
].copy()

table_04 = table_04.rename(
    columns={
        "Region": "Sub-grid",
        "Thematic_Mask": "Thematic Mask",
        "Group": "GHSL Range",
        "tau": "τ (%)",
        "composite_score": "Composite Score",
        "rmse": "RMSE",
        "n_days": "N days",
        "coverage_pct": "Temporal Coverage (%)",
    }
)

table_04["Sub-grid"] = pd.Categorical(
    table_04["Sub-grid"],
    categories=REGION_ORDER,
    ordered=True,
)

table_04 = table_04.sort_values("Sub-grid").reset_index(drop=True)

table_04_display = table_04.copy()
table_04_display["Composite Score"] = table_04_display["Composite Score"].round(3)
table_04_display["r"] = table_04_display["r"].round(2)
table_04_display["RMSE"] = table_04_display["RMSE"].round(2)
table_04_display["Temporal Coverage (%)"] = table_04_display["Temporal Coverage (%)"].round(2)

table_04_display

In [ ]:
# ==============================
# Export Main Table 4
# ==============================

table_04_display.to_csv(
    OUTPUT_MAIN_TABLE_DIR / "table_04_cross_subgrid_best_configurations.csv",
    index=False,
)


### 9.3 Table discussion

The highest-scoring configurations are concentrated in urban-focused masks. This is the most consistent transferable result. G1 or G2 is selected in every sub-grid, indicating that compact, persistently illuminated settlement classes provide the most defensible support for electricity-load alignment.

The selected spatial completeness threshold is not universal. Lower thresholds are selected in Samar–Leyte, Negros, and Panay, Bohol selects a slightly stricter threshold, and Cebu selects a substantially stricter threshold. This supports the main argument that transferability lies in the diagnostic framework and selection logic, not in one fixed threshold.


## 10. Optional Transferability Summary Figure

### 10.1 Compact visual summary of Table 4

This figure is optional. Use it if you want a visual companion to Table 4 in presentations or supplementary materials.


In [ ]:
# ==============================
# Optional Transferability Summary Figure
# ==============================

def plot_transferability_summary(table_04_display):
    plot_df = table_04_display.copy()

    fig = go.Figure()

    fig.add_trace(
        go.Bar(
            x=plot_df["Sub-grid"].astype(str),
            y=plot_df["r"],
            name="Pearson r",
            marker_color="#4E79A7",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=plot_df["Sub-grid"].astype(str),
            y=plot_df["Temporal Coverage (%)"],
            name="Temporal coverage (%)",
            mode="lines+markers",
            yaxis="y2",
            line=dict(color="black", width=2),
            marker=dict(size=7),
        )
    )

    for _, row in plot_df.iterrows():
        fig.add_annotation(
            x=str(row["Sub-grid"]),
            y=row["r"],
            text=f"{row['Thematic Mask']}<br>τ={row['τ (%)']}%",
            showarrow=False,
            yshift=35,
            font=dict(size=11),
        )

    fig.update_layout(
        template="plotly_white",
        width=1000,
        height=520,
        xaxis=dict(title="Sub-grid"),
        yaxis=dict(title="Pearson r", range=[0, 1]),
        yaxis2=dict(title="Temporal coverage (%)", overlaying="y", side="right", range=[0, 60], showgrid=False),
        legend=dict(orientation="h", y=1.12, x=0.5, xanchor="center"),
        margin=dict(t=90, b=70, l=70, r=80),
        paper_bgcolor="rgba(0,0,0,0)",
    )

    return fig


fig_sx_transferability_summary = plot_transferability_summary(table_04_display)
fig_sx_transferability_summary.show()

In [ ]:
# ==============================
# Export Optional Supplementary Figure
# ==============================

export_plotly_figure(
    fig_sx_transferability_summary,
    OUTPUT_SUPP_FIG_DIR / "fig_sx_cross_subgrid_transferability_summary",
    width=1000,
    height=520,
    scale=3,
)

## 11. Notebook Outputs

### 11.1 Main manuscript output

| Output | Path |
|---|---|
| Table 4 — Highest-scoring reliability-qualified configurations for each Visayas sub-grid | `outputs/tables/main/table_04_cross_subgrid_best_configurations.csv` |

### 11.2 Supplementary outputs

| Output | Path |
|---|---|
| NTL input inventory | `outputs/tables/supplementary/table_sx_cross_subgrid_ntl_input_inventory.csv` |
| NGCP 1 AM load table | `outputs/tables/supplementary/table_sx_ngcp_1am_load_visayas_2012_2024.csv` |
| All cross-sub-grid alignment metrics | `outputs/tables/supplementary/table_sx_cross_subgrid_alignment_metrics_all.csv` |
| Ranked cross-sub-grid alignment metrics | `outputs/tables/supplementary/table_sx_cross_subgrid_alignment_metrics_ranked.csv` |
| Table S10 — Top five configurations per sub-grid | `outputs/tables/supplementary/table_s10_top5_cross_subgrid_alignment_configurations.csv` |
| NGCP sub-grid load figure | `outputs/figures/supplementary/fig_sx_ngcp_subgrid_load_ma30_2012_2024.*` |
| Region-specific NTL time-series viewer | `outputs/figures/supplementary/fig_sx_region_ntl_timeseries_samar_leyte.html` |
| Regional comparison by GHSL group | `outputs/figures/supplementary/fig_sx_region_comparison_by_ghsl_group.html` |
| Region-specific z-score alignment examples | `outputs/figures/supplementary/fig_sx_<region>_zscore_alignment_tau<τ>.*` |
| Optional transferability summary figure | `outputs/figures/supplementary/fig_sx_cross_subgrid_transferability_summary.*` |




## 12. Summary

This notebook tests whether the reliability-qualified NTL workflow generalises beyond the primary Samar–Leyte diagnostic region. The same processing sequence is applied to Bohol, Cebu, Negros, and Panay: GHSL-masked DNB-BRDF aggregation, 0–95th percentile filtering, spatial-completeness thresholding, 30-day smoothing, z-score normalisation, and alignment with NGCP 1 AM electricity load.

The strongest transferable pattern is settlement-based rather than threshold-based. Urban-focused masks are selected across all five sub-grids, but the highest-scoring spatial completeness threshold varies by island. This indicates that dense, persistently illuminated settlement classes provide the most stable NTL–load signal, while the admissible completeness threshold should be selected through an explicit local sensitivity sweep.

The resulting Table 4 is the main manuscript output from this notebook. The expanded Table S10 provides the full top-five configuration set for each sub-grid and shows that the performance surface is broad rather than defined by a single sharply optimal setting.
